
# Fandom Cohesion Pilot — 실제 데이터 병행 산출 및 검증

`docs/FANDOM_COHESION_PILOT.md`가 설명하듯, 원본 "팬덤결속 지수"(Fandom Cohesion Index,
5개 유형 A~E 키워드매칭, `fandom_cohesion_index_v7.json`)는 이번 세션에 존재하지 않는다.
차트 스크립트(`build_cohesion_index_v7.py`)에는 5개 유형의 이름·색상 스키마는 하드코딩돼
남아있지만, 실제 매칭 키워드 사전과 수치 결과(원본 JSON)는 없다.

대신 이번 세션에 실제로 복구된 `data/v6_r22_snapshot/fandom_scores_v6.json`에는 K=8/M=6
LDA 메타요인 분해 결과인 `factor_share`가 100개 팬덤 전원에 대해 이미 계산되어 있고, 그중
하나가 `docs/LDA_V6_V7_TECHNICAL_SPECIFICATION.md` 4장이 정의하는 F4 "결속형(팬클럽·기부·
커뮤니티)"(키워드: 공식·팬클럽·팬덤·데뷔·기부·2025년·활동·2026년)다.

Ad/Commercial Pilot 노트북과 동일한 논리로, 이 값을 **원본의 재현이 아니라 같은 개념(팬클럽·
기부·커뮤니티 결속)에 답하는 독립적인 병행 산출물**로 취급해 검증·분석한다.


In [1]:

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")
FACTOR = "결속형(팬클럽·기부·커뮤니티)"

with open(DATA_DIR / "fandom_scores_v6.json", encoding="utf-8") as f:
    scores = json.load(f)

print(f"fandom_scores_v6.json: {len(scores)}개 팬덤")
print("샘플(BTS) factor_share:", json.dumps(scores[0]["factor_share"], ensure_ascii=False, indent=2))


fandom_scores_v6.json: 100개 팬덤
샘플(BTS) factor_share: {
  "소비력형(초동·판매·앨범)": 0.27,
  "현장경제형(콘서트·투어·매진)": 0.292,
  "브랜드·상업형(광고·앰버서더)": 0.0472,
  "소비력형(초동·판매·앨범)·데뷔형": 0.1614,
  "결속형(팬클럽·기부·커뮤니티)": 0.0855,
  "미디어노출형(방송·조회수)": 0.1439
}



## 1. 데이터 무결성 검증

Ad/Commercial Pilot 노트북과 동일하게, `factor_share`가 6개 메타요인에 대해 합 1.0이
되는지, 그리고 `dominant_factor`가 실제로 `factor_share`의 최댓값과 일치하는지 100개 팬덤
전원에 대해 재확인한다.


In [2]:

sum_mismatch = []
dominant_mismatch = []

for d in scores:
    fname = d["fandom"]
    shares = d["factor_share"]
    s = sum(shares.values())
    if abs(s - 1.0) > 0.001:
        sum_mismatch.append((fname, round(s, 4)))

    recomputed_dominant = max(shares, key=shares.get)
    if recomputed_dominant != d["dominant_factor"]:
        dominant_mismatch.append((fname, recomputed_dominant, d["dominant_factor"]))

print(f"factor_share 합 != 1.0 인 팬덤 수: {len(sum_mismatch)} / {len(scores)}")
if sum_mismatch:
    print("  불일치 목록:", sum_mismatch[:10])

print(f"dominant_factor 재계산 불일치 팬덤 수: {len(dominant_mismatch)} / {len(scores)}")
if dominant_mismatch:
    print("  불일치 목록:", dominant_mismatch[:10])


factor_share 합 != 1.0 인 팬덤 수: 0 / 100
dominant_factor 재계산 불일치 팬덤 수: 0 / 100



## 2. "결속형(팬클럽·기부·커뮤니티)" 비중 — 원본 "팬덤결속 지수"에 대응하는 병행 지표

원본 차트 우측 패널의 "팬덤별 결속 유형 구성·비중"에 대응하는 값으로, F4 메타요인 비중을
100개 팬덤 전체에 대해 산출해 내림차순 정렬한다.


In [3]:

rows = []
for d in scores:
    rows.append({
        "팬덤": d["fandom"],
        "구분": d["category"],
        "근거문장수(activity)": d["activity"],
        "결속형 비중": round(d["factor_share"].get(FACTOR, 0.0), 4),
        "대표 메타요인(6개 중)": d["dominant_factor"],
        "대표 메타요인이 결속형인가": d["dominant_factor"] == FACTOR,
    })

coh_df = pd.DataFrame(rows).sort_values("결속형 비중", ascending=False).reset_index(drop=True)
coh_df.index = coh_df.index + 1
coh_df.head(15)


,팬덤,구분,근거문장수(activity),결속형 비중,대표 메타요인(6개 중),대표 메타요인이 결속형인가
1,김호중,트로트,65,0.2798,결속형(팬클럽·기부·커뮤니티),True
2,god,원로그룹,45,0.2613,현장경제형(콘서트·투어·매진),False
3,잭스키스,원로그룹,44,0.2527,현장경제형(콘서트·투어·매진),False
4,윤하,솔로,40,0.2369,현장경제형(콘서트·투어·매진),False
5,창모,힙합,22,0.2360,현장경제형(콘서트·투어·매진),False
6,정동원,트로트,63,0.2358,결속형(팬클럽·기부·커뮤니티),True
7,이찬원,트로트,58,0.2346,현장경제형(콘서트·투어·매진),False
8,보아,솔로,50,0.2288,현장경제형(콘서트·투어·매진),False
9,윤도현,록,30,0.2218,현장경제형(콘서트·투어·매진),False
10,영탁,트로트,57,0.2216,현장경제형(콘서트·투어·매진),False



## 3. 원본 차트가 강조했던 3개 팬덤 — BTS·임영웅·리센느(RESCENE) 대조

원본 차트(`build_cohesion_index_v7.py`)는 우측 패널에서 BTS·임영웅·리센느(RESCENE) 3개
팬덤의 y축 라벨을 굵게 강조했다. 원본 수치는 복구되지 않으므로 같은 방식으로 강조할 수는
없지만, 이번 세션의 병행 지표(F4 비중)로 이 3개 팬덤이 100개 팬덤 중 어디쯤에 위치하는지
직접 확인한다.


In [4]:

HIGHLIGHT = ["BTS", "임영웅", "리센느(RESCENE)"]

for name in HIGHLIGHT:
    match = coh_df[coh_df["팬덤"] == name]
    if match.empty:
        print(f"{name}: fandom_scores_v6.json에 없음")
        continue
    rank = match.index[0]
    share = match["결속형 비중"].iloc[0]
    print(f"{name}: 결속형 비중={share} (100개 팬덤 중 {rank}위)")


BTS: 결속형 비중=0.0855 (100개 팬덤 중 63위)
임영웅: 결속형 비중=0.1906 (100개 팬덤 중 12위)
리센느(RESCENE): 결속형 비중=0.1819 (100개 팬덤 중 13위)



## 4. 대표 메타요인이 "결속형"인 팬덤은 몇 개인가

원본의 "팬덤결속 신호가 있는 팬덤 수"(N/100)에 대응할 만한 값으로, 이 메타요인이 그
팬덤의 **가장 큰** 성격 요인인 경우를 확인한다.


In [5]:

n_dominant = (coh_df["대표 메타요인이 결속형인가"]).sum()
print(f"대표 메타요인(dominant_factor)이 '결속형(팬클럽·기부·커뮤니티)'인 팬덤: {n_dominant} / {len(coh_df)}")
print()
print("해당 팬덤:")
print(coh_df[coh_df["대표 메타요인이 결속형인가"]][["팬덤", "구분", "결속형 비중"]])


대표 메타요인(dominant_factor)이 '결속형(팬클럽·기부·커뮤니티)'인 팬덤: 2 / 100

해당 팬덤:
    팬덤   구분  결속형 비중
1  김호중  트로트  0.2798
6  정동원  트로트  0.2358



## 5. 카테고리별 평균 비중 — 어느 장르가 "결속형" 성향이 강한가

100개 팬덤을 `구분`(category)별로 묶어 "결속형" 평균 비중을 비교한다. 원로그룹·트로트처럼
활동 기간이 길고 충성도 높은 팬덤층에서 팬클럽·기부·커뮤니티 결속 신호가 더 강하게
나타나는지 실측으로 확인한다.


In [6]:

cat_df = (
    coh_df.groupby("구분")["결속형 비중"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "평균 비중", "count": "팬덤 수"})
    .sort_values("평균 비중", ascending=False)
)
cat_df["평균 비중"] = cat_df["평균 비중"].round(4)
cat_df


,평균 비중,팬덤 수
구분,,
원로그룹,0.2105,3
트로트,0.1829,9
록,0.1513,2
솔로,0.1297,18
발라드,0.1168,16
힙합,0.1087,10
K-pop 보이그룹,0.0924,15
K-pop 걸그룹,0.0868,23
K-pop 보이그룹 /록,0.0778,3



## 6. 한계 (문서에서 이미 밝힌 것 재확인)

1. 이 노트북이 산출한 "결속형 비중"은 원본 팬덤결속 지수의 "결속 유형 구성·비중"과
   **개념은 대응하지만 계산 방법이 다르다** — 원본은 5개 유형(A~E) 키워드 사전 매칭 기반
   정수 카운트, 이 노트북은 LDA 토픽 혼합비중이다.
2. **5개 유형(A~E)별 세부 구성은 전혀 복구되지 않았다** — "대표 메타요인"은 6개 메타요인
   중 하나를 가리킬 뿐, 원본의 A~E 5개 유형 중 어디에 해당하는지는 알 수 없다.
3. 원본이 제공하던 코퍼스 전체 커버리지 통계("N/100개 팬덤에서 1건 이상 등장" 등)는
   `factor_share`가 애초에 6개 메타요인 합이 1.0으로 정규화된 구조라 개념적으로 대응할
   수 없다 — 모든 팬덤이 6개 메타요인 각각에 대해 항상 0보다 큰 비중을 갖는다.
4. 코퍼스 규모(5,612건, round22)가 원본 v7 45라운드 시점의 코퍼스 규모와 다르므로, 이
   노트북의 표는 round22 기준의 **독립적인 병행 산출물**이지 원본 차트/CSV의 재현이 아니다.
